# Global Gold Price Forecasting & Intelligence Engine
## Research-Grade Multi-Horizon Machine Learning & Econometric Modeling (Task 02)

### Objectives:
1. **Leak-Free Feature Engineering**: Lags (1d, 5d, 10d, 20d, 30d), Rolling Means (7d, 30d, 90d), Rolling Volatilities (7d, 30d, 90d), USD/INR FX dynamics, and cyclical trigonometric seasonality.
2. **Econometric & ML Architectures**:
   - **Naive Random Walk Baseline** (Persistence martingale null hypothesis: $\hat{y}_{t+h} = y_t$)
   - **ARIMA(1,1,1)** (Linear autoregressive integrated moving average with analytical confidence intervals)
   - **Facebook Prophet** (Additive seasonal decomposition with piecewise trend & Bayesian intervals)
   - **XGBoost Multi-Horizon Direct Regressors** (Gradient boosted trees with SHAP interpretability)
3. **Chronological Backtesting & Validation**: Strict chronological train/validation/test split (2024–2026 Test Set) to prevent look-ahead bias.
4. **Multi-Horizon Forecasts**: 30-day, 90-day, 180-day, and 365-day point forecasts with 95% uncertainty intervals.
5. **Market Structure Analysis**: Bullish/Bearish/Sideways trend classification (SMA50 vs SMA200), historical volatility cones, and drawdown analytics.
6. **SHAP Feature Importance**: Explainability analysis for XGBoost decision trees.

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

from ml.gold.feature_engineering import GoldFeatureEngineer
from ml.gold.evaluation import ModelEvaluator
from ml.gold.train_models import (
    NaiveBaselineForecaster,
    ArimaForecaster,
    ProphetForecaster,
    XGBoostMultiHorizonForecaster
)
from ml.gold.forecasting import GoldForecastingEngine
from ml.gold.model_comparison import GoldModelComparisonEngine

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
print("All modeling and forecasting libraries loaded successfully!")

## 1. Load Cleaned Master Gold Dataset
Loading the standardized historical cross-market dataset from Task 01.

In [ ]:
df_master = pd.read_csv('../data/processed/gold_master_daily_unified.csv')
df_gold = df_master[df_master['gold_market_open']].sort_values('date').reset_index(drop=True)
df_gold['date'] = pd.to_datetime(df_gold['date'])

print(f"Loaded {len(df_gold):,} trading sessions from {df_gold['date'].min().date()} to {df_gold['date'].max().date()}.")
print(f"Latest Close: ${df_gold['close_usd_per_troy_oz'].iloc[-1]:,.2f} / troy ounce")
df_gold.tail()

## 2. Leak-Free Feature Engineering
Generating lags, rolling statistics, returns, and seasonal features with strictly zero lookahead.

In [ ]:
fe = GoldFeatureEngineer()
df_feat = fe.build_features(df_gold, drop_na=True)
print(f"Engineered {len(fe.feature_names)} features across {len(df_feat):,} observations.")
print("Feature columns:", fe.feature_names[:12], "...")
df_feat[['date', 'close_usd_per_troy_oz', 'lag_price_1d', 'rolling_mean_7d', 'rolling_vol_30d', 'fx_usdinr_clean', 'sma_cross_ratio']].tail()

## 3. Out-of-Sample Model Evaluation & Benchmarking
Comparing all models chronologically against the **Naive Random Walk Baseline** on the 2024–2026 Test Set.

In [ ]:
df_metrics = pd.read_csv('../data/processed/model_metrics.csv')
display(df_metrics[['model', 'horizon', 'mae', 'rmse', 'mape', 'directional_accuracy_pct', 'skill_score_pct', 'notes']])

In [ ]:
# Visualizing model comparison metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# RMSE comparison
colors = ['#2B5B84', '#E24A4A', '#2CA02C', '#D4AF37']
ax1.bar(df_metrics['model'], df_metrics['rmse'], color=colors, width=0.5)
ax1.set_title('Out-of-Sample Test RMSE ($) - 30-Day Forecast Horizon', fontweight='bold')
ax1.set_ylabel('RMSE (USD / ozt)')
ax1.set_xticks(range(len(df_metrics['model'])))
ax1.set_xticklabels(df_metrics['model'], rotation=15, ha='right')
ax1.grid(True, linestyle='--', alpha=0.6)

# MAPE comparison
ax2.bar(df_metrics['model'], df_metrics['mape'], color=colors, width=0.5)
ax2.set_title('Mean Absolute Percentage Error (MAPE %)', fontweight='bold')
ax2.set_ylabel('MAPE (%)')
ax2.set_xticks(range(len(df_metrics['model'])))
ax2.set_xticklabels(df_metrics['model'], rotation=15, ha='right')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 4. Multi-Horizon Forecasts (30d, 90d, 180d, 365d) & 95% Prediction Intervals
Loading multi-horizon forecasts generated by the unified forecasting engine.

In [ ]:
df_preds = pd.read_csv('../data/processed/gold_predictions.csv')
display(df_preds[['model', 'horizon_days', 'current_price_usd', 'point_forecast_usd', 'lower_bound_95_usd', 'upper_bound_95_usd', 'forecast_return_pct']])

In [ ]:
# Plot Multi-Horizon Point Forecasts with Prediction Interval Bands
fig, ax = plt.subplots(figsize=(15, 6))
current_p = df_preds['current_price_usd'].iloc[0]

models = df_preds['model'].unique()
model_colors = {'Naive Random Walk': '#7f7f7f', 'ARIMA': '#1f77b4', 'Prophet': '#2ca02c', 'XGBoost': '#d62728'}

ax.axhline(current_p, color='black', linestyle='--', linewidth=1.5, label=f'Current Spot: ${current_p:,.2f}')

for m in models:
    sub = df_preds[df_preds['model'] == m].sort_values('horizon_days')
    x = sub['horizon_days']
    y = sub['point_forecast_usd']
    y_low = sub['lower_bound_95_usd']
    y_up = sub['upper_bound_95_usd']
    
    col = model_colors.get(m, '#333333')
    ax.plot(x, y, marker='o', label=f'{m} Point Forecast', color=col, linewidth=2)
    ax.fill_between(x, y_low, y_up, color=col, alpha=0.12)

ax.set_title('Gold Multi-Horizon Point Forecasts & 95% Prediction Intervals (30d to 365d)', fontsize=14, fontweight='bold')
ax.set_xlabel('Forecast Horizon (Trading Days)', fontweight='bold')
ax.set_ylabel('Forecast Price (USD / Troy Ounce)', fontweight='bold')
ax.set_xticks([30, 90, 180, 365])
ax.legend(loc='upper left', frameon=True)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Technical Trend, Volatility, and Drawdown Analytics

In [ ]:
comp_engine = GoldModelComparisonEngine()
analysis = comp_engine.compute_market_analysis(df_feat)

print("=== REAL-TIME MARKET STRUCTURE ANALYSIS ===")
print(f"Observation Date: {analysis['latest_date']}")
print(f"Spot Price: ${analysis['current_price_usd']:,.2f}/oz")
print(f"Trend Classification: {analysis['trend']['classification']} ({analysis['trend']['description']})")
print(f"SMA 50: ${analysis['trend']['sma_50d']:,.2f} | SMA 200: ${analysis['trend']['sma_200d']:,.2f} | Ratio: {analysis['trend']['sma_ratio']:.4f}")
print(f"30-Day Annualized Volatility: {analysis['volatility']['rolling_vol_30d_pct']:.2f}% ({analysis['volatility']['historical_vol_percentile']:.1f}th percentile)")
print(f"All-Time High: ${analysis['drawdown']['all_time_high_usd']:,.2f} on {analysis['drawdown']['all_time_high_date']}")
print(f"Current Drawdown: {analysis['drawdown']['current_drawdown_pct']:.2f}%")
print(f"Historical Max Drawdown: {analysis['drawdown']['historical_max_drawdown_pct']:.2f}%")

## 6. SHAP TreeExplainer Feature Importance (XGBoost)
Evaluating feature contributions across the gradient-boosted decision trees.

In [ ]:
df_fi = pd.read_csv('../data/processed/feature_importance.csv')

plt.figure(figsize=(12, 6))
top_fi = df_fi.head(15)
plt.barh(top_fi['feature'][::-1], top_fi['importance_mean_abs_shap'][::-1], color='#D4AF37')
plt.title('Top 15 Most Influential Features (XGBoost SHAP Feature Importance)', fontsize=13, fontweight='bold')
plt.xlabel('Mean |SHAP Value| (Impact on Forecast Price)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 7. FastAPI Endpoints Verification
Demonstrating programmatic retrieval through the FastAPI application.

In [ ]:
from fastapi.testclient import TestClient
from src.api.app import app

client = TestClient(app)

# Test GET /api/gold/trends
res_trends = client.get('/api/gold/trends')
print('GET /api/gold/trends Status:', res_trends.status_code)
print(res_trends.json()['market_analysis']['trend'])

# Test GET /api/gold/forecast
res_forecast = client.get('/api/gold/forecast?horizon_days=30')
print('\nGET /api/gold/forecast (30d) Status:', res_forecast.status_code)
for f in res_forecast.json()['forecasts']:
    print(f"  {f['model']:<20}: Point=${f['point_forecast_usd']:,.2f} [95% CI: ${f['lower_bound_95_usd']:,.2f} - ${f['upper_bound_95_usd']:,.2f}]")